## This extraction pipeline only work with 1 pdf at a time, re-run it 14 more times for a full extractions of 15 pdfs.

#1. Environment Setup



In [ ]:
!pip install pymupdf
!pip install --upgrade langchain-graphrag
!pip install graspologic
!pip install --upgrade langchain langchain-openai langchain-community langchain-experimental langchain-ollama
!pip install neo4j python-dotenv networkx json-repair

In [ ]:
import numpy as np
import pandas as pd
from langchain_core.documents import Document
from IPython import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import re

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#2. Text Extraction

Extracting text from ADA diabetes guideline pdfs. References and footnotes will be stripped off, leaving only the main content.

In [ ]:
### 1. Extract Text
print("\n##### EXTRACTING TEXT FROM PDF #####\n")
import fitz  # pymupdf

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc):
        text = page.get_text()
        pages.append({"page": page_num + 1, "text": text})
    return pages

_ls = extract_text("/content/drive/My Drive/genaiproj/papers/ADA_2026_chapter_16.pdf")

print(f"{len(_ls)} number of pages extracted.")

### 2. Remove Reference
print("\n##### EXTRACTING TEXT FROM PDF #####\n")

_p = re.compile(
    r'\nReferences\n'
    r'[\w\W]*'
)

idx = len(_ls) # Initialize idx to the total number of pages

for _dict in _ls[::-1]:
    if _p.search(_dict['text']):
        idx = _dict['page']
        break

_ls = _ls[:idx]
print(f"After removing Referencs, {len(_ls)} number of pages extracted left.")

print("BEFORE REMOVING REFERENCE")
print('='*100)
_ls[-1]['text']
_ls[-1]['text'] = _p.sub('', _ls[-1]['text'])
print("AFTER REMOVING REFERENCE")
print(f"Removed references content in the last exisitng page.")
print('='*100)
_ls[-1]['text']


##### EXTRACTING TEXT FROM PDF #####

17 number of pages extracted.

##### EXTRACTING TEXT FROM PDF #####

After removing Referencs, 13 number of pages extracted left.
BEFORE REMOVING REFERENCE


'we deliver in hospitals and in transitions \nfrom inpatient to outpatient.\nReferences\n1. Seisa MO, Saadi S, Nayfeh T, et al. A systematic \nreview supporting the endocrine society clinical \npractice guideline for the management of \nhyperglycemia in adults hospitalized for noncritical \nillness or undergoing elective surgical procedures. J \nClin Endocrinol Metab 2022;107:2139–2147\n2. Umpierrez GE, Isaacs SD, Bazargan N, You X, \nThaler LM, Kitabchi AE. Hyperglycemia: an \nindependent marker of in-hospital mortality in \npatients with undiagnosed diabetes. J Clin Endocrinol \nMetab 2002;87:978–982\n3. Pasquel FJ, Gomez-Huelgas R, Anzola I, et al. \nPredictive value of admission hemoglobin A1c on \ninpatient glycemic control and response to insulin \ntherapy in medicine and surgery patients with type 2 \ndiabetes. Diabetes Care 2015;38:e202-203–e203\n4. Umpierrez GE, Reyes D, Smiley D, et al. Hospital \ndischarge algorithm based on admission HbA1c for \nthe management of patients w

AFTER REMOVING REFERENCE
Removed references content in the last exisitng page.


'we deliver in hospitals and in transitions \nfrom inpatient to outpatient.'

In [ ]:
print("\n##### 3. Clean Text #####\n")

odd_footer = re.compile(
    r'diabetesjournals\.org/care '
    r'.+? '
    r'S\d+ '
    r'Downloaded from https?://\S+\.pdf '
    r'by guest on \d{1,2} \w+ \d{4}'
)

even_footer = re.compile(
    r'S\d+ '
    r'.+? '
    r'Diabetes Care Volume \d+, Supplement \d+, \w+ \d{4} '
    r'Downloaded from https?://\S+\.pdf '
    r'by guest on \d{1,2} \w+ \d{4}'
)

def clean_text(text):
    # Fix soft hyphens (shy hyphens) - PDF line-break artifacts
    text = text.replace('\xad', '')

    # Join lines that were broken mid-sentence
    # (lowercase letter or comma before \n, lowercase after)
    text = re.sub(r'(?<=[a-z,])\n(?=[a-z])', '', text)

    # Keep paragraph breaks (double newlines) as they signal real structure
    text = re.sub(r'\n{2,}', '\n\n', text)

    # Clean up remaining single newlines
    text = re.sub(r'\n', ' ', text)

    # Collapse multiple spaces
    text = re.sub(r' +', ' ', text)

    # remove footer
    # Remove odd page footer
    text = odd_footer.sub('', text)

    # Remove even page footer
    text = even_footer.sub('', text)

    return text.strip()

ls_of_txt = [clean_text(_dict['text']) for _dict in _ls]

print("Text cleaned.")

print("\n##### 4. CHUNKING #####\n")

full_text = '\n\n'.join(ls_of_txt)


##### 3. Clean Text #####

Text cleaned.

##### 4. CHUNKING #####



In [ ]:
full_text

'16. DIABETES CARE IN THE HOSPITAL 16. Diabetes Care in the Hospital: Standards of Care in Diabetes— 2026 Diabetes Care 2026;49(Suppl. 1):S339–S355 | https://doi.org/10.2337/dc26-S016 American Diabetes Association Professional Practice Committee for Diabetes* *A complete list of members of the American Diabetes Association Professional Practice Committee for Diabetes can be found at https://doi.org/ 10.2337/dc26-SINT. Duality of interest information for each contributor is available at https://doi.org/10.2337/dc26-SDIS. Suggested citation: American Diabetes Association Professional Practice Committee for Diabetes. 16. Diabetes care in the hospital: Standards of Care in Diabetes—2026. Diabetes Care 2026;49(Suppl. 1): S339–S355 © 2025 by the American Diabetes Association. Readers may use this work for educational, noncommercial purposes if properly cited and unaltered. The version of record may be linked at https://diabetesjournals.org/care, but ADA permission is required to post this wo

#3. LLM configuration

In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Fetched 388 kB in 2s (242 kB/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
128 packages can be upgraded. Run 'apt list --upgradable' to see th

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [ ]:
!ollama pull llama3:8b
!ollama list


NAME         ID              SIZE      MODIFIED               
llama3:8b    365c0bd3c000    4.7 GB    Less than a second ago    


In [ ]:
import os
from langchain_ollama import ChatOllama
from langchain_community.cache import SQLiteCache

llm = ChatOllama(model='llama3:8b', temperature=0, seed=42)

### Patching langchain package
Due to langchain-graphrag and graspologic use an older version of langchain, therefore some modules such as langchain.output-parser were not yet exist, thus this cell below patch the error.

In [ ]:
# Path to the problematic file within langchain_graphrag
file_to_patch = '/usr/local/lib/python3.12/dist-packages/langchain_graphrag/indexing/report_generation/_output_parser.py'

# Check if the file exists
if os.path.exists(file_to_patch):
    print(f"Patching: {file_to_patch}")
    with open(file_to_patch, 'r') as f:
        content = f.read()

    # Replace the old import with the new one
    # Only patch if the old import line is present and the new one is not already there
    old_import = 'from langchain.output_parsers import PydanticOutputParser'
    new_import = 'from langchain_core.output_parsers import PydanticOutputParser'

    if old_import in content and new_import not in content:
        content = content.replace(old_import, new_import)
        with open(file_to_patch, 'w') as f:
            f.write(content)
        print("Successfully patched langchain_graphrag for output_parsers compatibility.")
    elif new_import in content:
        print("Patch already applied.")
    else:
        print("Old import line not found, no patch applied (or file structure changed unexpectedly).")
else:
    print(f"File not found: {file_to_patch}. Manual patching may be required or the environment is different.")

Patching: /usr/local/lib/python3.12/dist-packages/langchain_graphrag/indexing/report_generation/_output_parser.py
Patch already applied.


#4. Splitting and processing entities and relationships and save the graph

In [ ]:
# Splitting text and extracting text unit
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_graphrag.indexing import TextUnitExtractor

#
doc = [Document(page_content=full_text, metadata={"source": "ADA_2026_chapter_16"})]

# Splitting text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)

# Extracting text units
text_unit_extractor = TextUnitExtractor(text_splitter=text_splitter)
df_text_unit = text_unit_extractor.run(doc)
df_text_unit

Processing documents ...: 100%|██████████| 1/1 [00:00<00:00, 35.64it/s]


,document_id,id,text_unit
0,50ca394c-58e6-4c67-affb-294df1784189,02dbf247-40b3-4335-8957-35004b591973,16. DIABETES CARE IN THE HOSPITAL 16. Diabetes...
1,50ca394c-58e6-4c67-affb-294df1784189,8236b92c-94c3-4c6b-9d00-2f3083f8f12f,.org. More information is available at https:/...
2,50ca394c-58e6-4c67-affb-294df1784189,02105476-dd75-4748-9eb9-a86a86c877bf,and treatment of hyperglycemia and hypoglycemi...
3,50ca394c-58e6-4c67-affb-294df1784189,163eb9f9-506a-4c05-a577-916370937735,because inpatient treatment and discharge plan...
4,50ca394c-58e6-4c67-affb-294df1784189,7e0a28cd-bc91-4a6f-b218-62e55dce9502,quality improvement strategies for process imp...
...,...,...,...
56,50ca394c-58e6-4c67-affb-294df1784189,b2d45d9f-35ea-4946-a8d2-c96c93b7726e,can assist outpatient health care professional...
57,50ca394c-58e6-4c67-affb-294df1784189,b78723f3-87c4-487e-8f7c-45b1c617b997,and diabetes care after discharge. • Assess th...
58,50ca394c-58e6-4c67-affb-294df1784189,90a67419-0fb7-4335-8dcd-c7d711cb3193,"have two or more days of hospital stays, and t..."
59,50ca394c-58e6-4c67-affb-294df1784189,4f80e67a-35bc-48a4-8fd8-ad05896729cf,have shown that use of CGM may prevent emergen...


In [ ]:
# Using Llama3:8b to extracting entities and relationship
from langchain_graphrag.indexing.graph_generation import EntityRelationshipExtractor
extractor = EntityRelationshipExtractor.build_default(llm=llm)
graphs = extractor.invoke(df_text_unit)

Extracting entities and relationships ...: 100%|██████████| 61/61 [19:59<00:00, 19.66s/it]


In [ ]:
# Saving graph to folder
import pickle
import os

spath = '/content/drive/MyDrive/genaiproj/graph_save'
save_path = os.path.join(spath, 'graphs16.pkl')

with open(save_path, 'wb') as f:
    pickle.dump(graphs, f)

print(f'SAVED ! :{save_path}')

SAVED ! :/content/drive/MyDrive/genaiproj/graph_save/graphs16.pkl
